## Model Training - Random Forest Classifier

In [54]:
# import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
import ydf
import matplotlib.pyplot as plt
# read data
debris_df = pd.read_csv("../Data/DS_combined_final.csv")
debris_df["LIFETIME_CLASS"] = debris_df["LIFETIME_CLASS"].astype("category")

In [55]:
# split the data into training and test sets (80% training, 20% test)
train_df, test_df = train_test_split(debris_df, test_size=0.2, random_state=42)

print("Training set size:", len(train_df))
print("Test set size:", len(test_df))

Training set size: 19359
Test set size: 4840


#### Model 1 - using all features

In [ ]:
# train model
# not use columns that equal the label (-> if decay and launch both kept as features) or are unneccessary
features1 = ['OBJECT_TYPE', 'COUNTRY', 'SITE', 'LAUNCH', 
            'PERIAPSIS', 'APOAPSIS', 'ARG_OF_PERICENTER', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'MEAN_ANOMALY', 'REV_AT_EPOCH', 'BSTAR', 'MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT', 'SEMIMAJOR_AXIS', 'PERIOD']
model1 = ydf.RandomForestLearner(label="LIFETIME_CLASS", features = features1).train(train_df, verbose = 2)

Train model on 19359 examples
Model trained in 0:00:14.431815


In [24]:
model1.describe() # show model description

In [25]:
# Evaluate the model
model1.evaluate(test_df)

Label \ Pred,Class 1,Class 2,Class 3
Class 1,1579,56,14
Class 2,36,1509,92
Class 3,2,57,1495


In [26]:
model1.analyze(test_df) # analyze the test dataset

In [18]:
# save model
model1.save("../Models/RF_Model1")

In [3]:
# load model 1
model1 = ydf.load_model("../Models/RF_Model1")

#### Model 2 - using all features except REV_AT_EPOCH and LAUNCH since they are basically measures for the age of the object and it are highly correlated with the lifetime. That is beceause for every object the latest OMM file is used where the Epoch is mostly very near the decay date.

In [29]:
features2 = ['OBJECT_TYPE', 'COUNTRY', 'SITE',
            'PERIAPSIS', 'APOAPSIS', 'ARG_OF_PERICENTER', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'MEAN_ANOMALY', 'BSTAR', 'MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT', 'SEMIMAJOR_AXIS', 'PERIOD']
model2 = ydf.RandomForestLearner(label="LIFETIME_CLASS", features=features2).train(train_df, verbose = 2) # label: defines the prediction column; train_ds: defines the training dataset, verbose: show training logs

Train model on 19359 examples
Model trained in 0:00:03.197339


In [30]:
model2.describe() # show model description

In [31]:
# Evaluate the model
model2.evaluate(test_df)

Label \ Pred,Class 1,Class 2,Class 3
Class 1,1458,196,35
Class 2,131,1290,226
Class 3,28,136,1340


In [32]:
# analyze model
model2.analyze(test_df) # analyze the test dataset

In [33]:
# save model
model2.save("../Models/RF_Model2")

#### Model 3 - using only umerical features except REV_AT_EPOCH

In [35]:
features3 = ['PERIAPSIS', 'APOAPSIS', 'ARG_OF_PERICENTER', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'MEAN_ANOMALY', 'BSTAR', 'MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT', 'SEMIMAJOR_AXIS', 'PERIOD']
model3 = ydf.RandomForestLearner(label="LIFETIME_CLASS", features=features3).train(train_df, verbose = 2) # label: defines the prediction column; train_ds: defines the training dataset, verbose: show training logs

Train model on 19359 examples
Model trained in 0:00:03.535478


In [36]:
# model description
model3.describe() # show model description


In [37]:
# Evaluate the model
model3.evaluate(test_df) 

Label \ Pred,Class 1,Class 2,Class 3
Class 1,1408,219,46
Class 2,156,1202,272
Class 3,53,201,1283


In [38]:
# analyze model
model3.analyze(test_df) 

In [39]:
# save model
model3.save("../Models/RF_Model3")

#### Model 4 - using normalized features

In [ ]:
# normalize features using z normalization
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
# scale all columns except the target column
debris_df_scaled = debris_df.copy()
debris_df_scaled.iloc[:, 4:-1] = scaler.fit_transform(debris_df.iloc[:, 4:-1])

In [46]:
# split the data into training and test sets (80% training, 20% test)
train_df_scaled, test_df_scaled = train_test_split(debris_df_scaled, test_size=0.2, random_state=42)

print("Training set size:", len(train_df_scaled))
print("Test set size:", len(test_df_scaled))

Training set size: 19359
Test set size: 4840


In [49]:
# train model
features4 = ['PERIAPSIS', 'APOAPSIS', 'ARG_OF_PERICENTER', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'MEAN_ANOMALY', 'BSTAR', 'MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT', 'SEMIMAJOR_AXIS', 'PERIOD']
model4 = ydf.RandomForestLearner(label="LIFETIME_CLASS", features=features4).train(train_df_scaled, verbose = 2) # label: defines the prediction column; train_ds: defines the training dataset, verbose: show training logs

Train model on 19359 examples
Model trained in 0:00:03.490843


In [50]:
# see model stats
model4.describe()

In [51]:
# Evaluate the model
model4.evaluate(test_df)

Label \ Pred,Class 1,Class 2,Class 3
Class 1,0,0,0
Class 2,5,2,5
Class 3,1612,1620,1596


In [52]:
# analyze model
model4.analyze(test_df)

In [53]:
# save model
model4.save("../Models/RF_Model4")

#### Model 5 - using only Keplerian Elements

In [56]:
features5 = ['ARG_OF_PERICENTER', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'MEAN_ANOMALY', 'SEMIMAJOR_AXIS']
model5 = ydf.RandomForestLearner(label="LIFETIME_CLASS", features=features5).train(train_df, verbose = 2) # label: defines the prediction column; train_ds: defines the training dataset, verbose: show training logs

Train model on 19359 examples
Model trained in 0:00:02.834967


In [57]:
# see model stats
model5.describe()

In [58]:
# Evaluate the model
model5.evaluate(test_df)

Label \ Pred,Class 1,Class 2,Class 3
Class 1,1384,207,71
Class 2,168,1156,337
Class 3,65,259,1193


In [59]:
# analyze model
model5.analyze(test_df)

In [60]:
# save model
model5.save("../Models/RF_Model5")